In [ ]:
!pip uninstall -y protobuf tensorflow
!pip install protobuf==3.20.3 tensorflow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import json

print("✓ All libraries imported successfully!")

In [ ]:
import cv2

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
keras.utils.set_random_seed(42)

# Configuration parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
DATA_DIR = '/kaggle/input/d/abhyudayapokhrel/civic-issue-dataset/Dataset'

CLASSES_TO_USE = ['Garbage', 'Potholes', 'NoIssue']

TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

print(f"Configuration:")
print(f"  Image Size: {IMG_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Data Directory: {DATA_DIR}")
print(f"  Classes to train: {CLASSES_TO_USE}")

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)

# Only rescaling for validation
val_test_datagen = ImageDataGenerator(
    rescale=1./255
)

print("✓ Data generators created with augmentation settings")
print("  Training: Augmentation enabled")
print("  Validation & Test: Only rescaling")

In [ ]:
# Create temporary directories for train/val/test split
TEMP_DIR = '/kaggle/working/temp_data'
TRAIN_DIR = os.path.join(TEMP_DIR, 'train')
VAL_DIR = os.path.join(TEMP_DIR, 'val')
TEST_DIR = os.path.join(TEMP_DIR, 'test')

# Clean up if exists
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

# Create directories
for class_name in CLASSES_TO_USE:
    os.makedirs(os.path.join(TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(VAL_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, class_name), exist_ok=True)

print("Splitting data into train/val/test sets...")

# Split data for each class
for class_name in CLASSES_TO_USE:
    class_dir = os.path.join(DATA_DIR, class_name)
    
    # Get all image files
    image_files = [f for f in os.listdir(class_dir) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    # First split: separate test set
    train_val_files, test_files = train_test_split(
        image_files, 
        test_size=TEST_SPLIT, 
        random_state=42
    )
    
    # Second split: separate train and validation
    train_files, val_files = train_test_split(
        train_val_files,
        test_size=VAL_SPLIT/(TRAIN_SPLIT + VAL_SPLIT),  # Adjust ratio
        random_state=42
    )
    
    # Copy files to respective directories
    for filename in train_files:
        src = os.path.join(class_dir, filename)
        dst = os.path.join(TRAIN_DIR, class_name, filename)
        shutil.copy2(src, dst)
    
    for filename in val_files:
        src = os.path.join(class_dir, filename)
        dst = os.path.join(VAL_DIR, class_name, filename)
        shutil.copy2(src, dst)
    
    for filename in test_files:
        src = os.path.join(class_dir, filename)
        dst = os.path.join(TEST_DIR, class_name, filename)
        shutil.copy2(src, dst)
    
    print(f"  {class_name}: {len(train_files)} train, {len(val_files)} val, {len(test_files)} test")

print("\n✓ Data split completed!")

# Load training data
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # Changed to categorical for 3 classes
    shuffle=True
)

# Load validation data
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Load test data
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n✓ Data generators created!")
print(f"  Training images: {train_generator.samples}")
print(f"  Validation images: {val_generator.samples}")
print(f"  Test images: {test_generator.samples}")
print(f"  Class indices: {train_generator.class_indices}")
print(f"  Number of classes: {train_generator.num_classes}")

In [ ]:
model = keras.Sequential([
    # First convolutional block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Second convolutional block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Third convolutional block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Fourth convolutional block
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Flatten and dense layers
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')
])

print("✓ Model architecture created")
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'recall', 'precision']
)

print("\n✓ Model compiled successfully!")
print("  Optimizer: Adam (lr=0.001)")
print("  Loss: Categorical Crossentropy (for 3 classes)")
print("  Metrics: Accuracy, Recall, Precision")

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Callbacks configured:")
print("  - Early Stopping (patience=5)")
print("  - Learning Rate Reduction (patience=3)")

In [ ]:
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50 + "\n")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*50)
print("TRAINING COMPLETED!")
print("="*50)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss', marker='o')
ax2.plot(history.history['val_loss'], label='Val Loss', marker='s')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"\nFinal Training Accuracy: {final_train_acc*100:.2f}%")
print(f"Final Validation Accuracy: {final_val_acc*100:.2f}%")
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")

In [ ]:
print("\n" + "="*50)
print("EVALUATING MODEL ON VALIDATION SET")
print("="*50 + "\n")

val_results = model.evaluate(val_generator, verbose=1)

print()
for name, value in zip(model.metrics_names, val_results):
    if "accuracy" in name:
        print(f"✓ Validation {name.capitalize()}: {value*100:.2f}%")
    else:
        print(f"✓ Validation {name.capitalize()}: {value:.4f}")

print("\n" + "="*50)
print("EVALUATING MODEL ON TEST SET")
print("="*50 + "\n")

test_results = model.evaluate(test_generator, verbose=1)

print()
for name, value in zip(model.metrics_names, test_results):
    if "accuracy" in name:
        print(f"✓ Test {name.capitalize()}: {value*100:.2f}%")
    else:
        print(f"✓ Test {name.capitalize()}: {value:.4f}")


In [ ]:
print("\n" + "="*50)
print("GENERATING PREDICTIONS ON TEST SET")
print("="*50 + "\n")

# Generate predictions on test set
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)  # Get class with highest probability
y_true = test_generator.classes

# Get class names
class_names = list(train_generator.class_indices.keys())

# Classification report
print("\n" + "="*50)
print("CLASSIFICATION REPORT (TEST SET)")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix (Test Set)', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-Class Accuracy:")
for i, class_name in enumerate(class_names):
    class_mask = y_true == i
    class_acc = (y_pred[class_mask] == y_true[class_mask]).sum() / class_mask.sum()
    print(f"  {class_name}: {class_acc*100:.2f}%")

In [ ]:
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(f"{SAVE_DIR}/pothole_garbage_classifier.h5")
model.save(f"{SAVE_DIR}/pothole_garbage_classifier.keras")

class_info = {
    'class_names': class_names,
    'class_indices': train_generator.class_indices,
    'num_classes': len(class_names),
    'img_size': IMG_SIZE
}

with open(f"{SAVE_DIR}/class_info.json", "w") as f:
    json.dump(class_info, f, indent=4)

print("✓ Model and metadata saved to /kaggle/working")

In [ ]:
def predict_for_single_image(img_path, show_plot=True):
    """
    Predict class of a single image
    
    Args:
        img_path: Path to the image file
        show_plot: Whether to display the image with prediction
    
    Returns:
        tuple: (class_name, confidence, all_probabilities)
    """
    # Load and preprocess image
    img = keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    # Make prediction
    predictions = model.predict(img_array, verbose=0)[0]
    class_idx = np.argmax(predictions)
    class_name = class_names[class_idx]
    confidence = predictions[class_idx]
    
    # Display result
    if show_plot:
        plt.figure(figsize=(10, 6))
        
        # Image subplot
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title(f'Prediction: {class_name.upper()}\nConfidence: {confidence*100:.2f}%', 
                  fontsize=12, fontweight='bold')
        plt.axis('off')
        
        # Probability bar chart
        plt.subplot(1, 2, 2)
        colors = ['green' if i == class_idx else 'gray' for i in range(len(class_names))]
        plt.barh(class_names, predictions * 100, color=colors)
        plt.xlabel('Confidence (%)', fontsize=10)
        plt.title('Class Probabilities', fontsize=12, fontweight='bold')
        plt.xlim(0, 100)
        
        plt.tight_layout()
        plt.show()
    
    return class_name, confidence, predictions

print("✓ Prediction function defined")
print("\nUsage example:")
print("  predict_image('/path/to/image.jpg')")
print("\nReturns: (class_name, confidence, all_probabilities)")



In [ ]:
# Evaluate the model to get the test accuracy
results = model.evaluate(test_generator, verbose=1)

# results[0] is usually loss, results[1] is usually accuracy
test_acc = results[1] 

# Now your print statement will work!
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
print("\n" + "="*50)
print("SAMPLE PREDICTIONS ON TEST DATA")
print("="*50 + "\n")

test_generator.reset()
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    batch_size=32,
    shuffle=True) # This ensures a mix of Potholes and Garbage
x_batch, y_batch = next(test_generator)
predictions_batch = model.predict(x_batch, verbose=0)

plt.figure(figsize=(15, 10))
for i in range(min(12, len(x_batch))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_batch[i])
    
    true_class_idx = np.argmax(y_batch[i])
    pred_class_idx = np.argmax(predictions_batch[i])
    
    true_class = class_names[true_class_idx]
    pred_class = class_names[pred_class_idx]
    confidence = predictions_batch[i][pred_class_idx]
    
    # Color code: green if correct, red if wrong
    color = 'green' if true_class == pred_class else 'red'
    
    plt.title(f'True: {true_class}\nPred: {pred_class} ({confidence*100:.1f}%)', 
              fontsize=9, color=color, fontweight='bold')
    plt.axis('off')

plt.suptitle('Sample Predictions on Test Set (Green=Correct, Red=Incorrect)', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("ALL BLOCKS COMPLETED SUCCESSFULLY!")
print("="*50)
print(f"\nFinal Results Summary:")
print(f"  Classes: {class_names}")
print(f"  Test Accuracy: {test_acc*100:.2f}%")
print(f"  Model saved as: pothole_garbage_classifier.h5")

In [ ]:
print(f"For i1 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/i1.png')}")
print(f"For i11 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/i11.jpeg')}")
print(f"For i2 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/i2.png')}")
print(f"For i3 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/i3.png')}")
print(f"For n1 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/n1.jpg')}")
print(f"For n2 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/n2.jpg')}")
print(f"For n3 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/n3.png')}")
print(f"For n4 we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/n4.jpeg')}")
print(f"For boomer we got {predict_for_single_image( '/kaggle/input/helpppp/helppp/n4.jpeg')}")

In [ ]:
# 1. Create the directory if it doesn't exist
model_dir = "/kaggle/working/models"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# 2. Save trained model into that folder
model.save(f"{model_dir}/pothole_garbage_classifier.h5")

# 3. Zip the folder
shutil.make_archive("/kaggle/working/models_download", 'zip', model_dir)

print("✅ Model saved and zipped successfully!")